# NGBoost + DiCE: Water Quality Classification
## Pipeline V3: Median Imputation + MinMax + SMOTE + Threshold Optimization

**Perbedaan dari V2:**
- V2: Threshold default 0.5
- V3: Threshold dioptimasi pada validation set untuk maximize F1-Score

**Teknik Baru:**
- Threshold Optimization: Mencari threshold keputusan optimal yang memaksimalkan F1-Score
- Ref: [11] Zhang et al. - "Threshold Moving Approaches for Addressing the Class Imbalance Problem"

**Referensi Pendukung:**
- [2] Salmi et al. 2024 - Handling imbalanced datasets
- [3] Abdelhamid & Desai 2024 - Class imbalance binary classification
- [7] Grinsztajn et al. 2022 - Tree-based models for tabular data
- [11] Zhang et al. - Threshold moving approaches
- [13] van de Mortel & van Wingen 2025 - Data leakage prevention

## Cell 1: Instalasi Library

In [ ]:
!pip install ngboost dice-ml xgboost scikit-learn pandas numpy matplotlib seaborn imbalanced-learn

## Cell 2: Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve,
    classification_report, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from ngboost import NGBClassifier
from ngboost.distns import Bernoulli

from xgboost import XGBClassifier

from scipy.stats import chi2

warnings.filterwarnings("ignore")
np.random.seed(42)

# Buat folder figures jika belum ada
os.makedirs("figures", exist_ok=True)

print("Semua library berhasil diimport!")

## Cell 3: Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv("water_potability.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
display(df.head())

print(f"\nDataset Info:")
df.info()

print(f"\nMissing Values:")
print(df.isnull().sum())

print(f"\nDistribusi Kelas (Potability):")
print(df["Potability"].value_counts())
print(f"\nPersentase:")
print(df["Potability"].value_counts(normalize=True) * 100)

## Cell 4: Stratified Split 70/15/15

Membagi dataset menjadi 3 partisi:
- Training set: 70%
- Validation set: 15% (untuk threshold optimization)
- Test set: 15% (untuk evaluasi final)

In [ ]:
# Pisahkan fitur dan label
X = df.drop("Potability", axis=1)
y = df["Potability"]

feature_names = X.columns.tolist()
print(f"Fitur ({len(feature_names)}): {feature_names}")

# Split pertama: 85% temp, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Split kedua: dari temp, 15/85 sebagai validation
# 15/85 ~ 0.1765 dari temp untuk mendapat ~15% dari total
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, random_state=42, stratify=y_temp
)

print("=" * 60)
print("DATA SPLIT RESULTS")
print("=" * 60)
print(f"Training set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set      : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total         : {len(X)} samples")

print(f"\nDistribusi kelas:")
print(f"  Train - Class 0: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%), Class 1: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")
print(f"  Val   - Class 0: {(y_val==0).sum()} ({(y_val==0).mean()*100:.1f}%), Class 1: {(y_val==1).sum()} ({(y_val==1).mean()*100:.1f}%)")
print(f"  Test  - Class 0: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%), Class 1: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)")

## Cell 5: Preprocessing + SMOTE (Zero Data Leakage)

Langkah preprocessing manual untuk NGBoost:
1. Median Imputation (fit pada train saja)
2. MinMax Scaling (fit pada train saja)
3. SMOTE hanya pada training set

Ref: [13] van de Mortel & van Wingen 2025 - Data leakage prevention

In [ ]:
# ====== PREPROCESSING MANUAL UNTUK NGBoost ======

# Step 1: Median Imputation
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

# Step 2: MinMax Scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Step 3: SMOTE hanya pada training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("=" * 60)
print("PREPROCESSING RESULTS")
print("=" * 60)
print(f"Sebelum SMOTE - Train: {X_train_scaled.shape[0]} samples")
print(f"  Class 0: {(y_train==0).sum()}, Class 1: {(y_train==1).sum()}")
print(f"\nSetelah SMOTE - Train: {X_train_resampled.shape[0]} samples")
print(f"  Class 0: {(y_train_resampled==0).sum()}, Class 1: {(y_train_resampled==1).sum()}")
print(f"\nValidation: {X_val_scaled.shape[0]} samples (tidak di-SMOTE)")
print(f"Test: {X_test_scaled.shape[0]} samples (tidak di-SMOTE)")

# ====== IMBLEARN PIPELINE UNTUK XGBoost DAN RF ======
# Pipeline ini otomatis handle SMOTE hanya saat fit

xgb_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False,
        verbosity=0
    ))
])

rf_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

print("\nImblearn Pipeline untuk XGBoost dan RF telah dibuat.")

## Cell 6: Training 3 Models

Melatih tiga model klasifikasi:
1. NGBoost (Natural Gradient Boosting)
2. XGBoost (Extreme Gradient Boosting)
3. Random Forest

In [ ]:
# ====== NGBoost ======
print("Training NGBoost...")
ngb_model = NGBClassifier(
    Dist=Bernoulli,
    n_estimators=300,
    learning_rate=0.05,
    minibatch_frac=0.8,
    col_sample=0.8,
    random_state=42,
    Base=DecisionTreeRegressor(max_depth=4),
    verbose=False
)

ngb_model.fit(
    X_train_resampled, y_train_resampled,
    X_val=X_val_scaled, Y_val=y_val
)
print("NGBoost training complete!")

# ====== XGBoost (via ImbPipeline) ======
print("\nTraining XGBoost...")
xgb_pipeline.fit(X_train, y_train)
print("XGBoost training complete!")

# ====== Random Forest (via ImbPipeline) ======
print("\nTraining Random Forest...")
rf_pipeline.fit(X_train, y_train)
print("Random Forest training complete!")

print("\n" + "=" * 60)
print("SEMUA MODEL BERHASIL DILATIH!")
print("=" * 60)

## Cell 7: Threshold Optimization (FITUR BARU V3)

Mencari threshold optimal yang memaksimalkan F1-Score pada validation set.
Threshold ini akan digunakan untuk evaluasi pada test set.

Ref: [11] Zhang et al. - "Threshold Moving Approaches for Addressing the Class Imbalance Problem"

**PENTING:** Threshold dicari pada VALIDATION set, bukan test set, untuk menghindari data leakage.

In [ ]:
def find_optimal_threshold(y_true, y_prob, metric="f1"):
    """
    Cari threshold yang memaksimalkan F1-Score pada validation set.
    Ref: [11] Zhang et al. - Threshold Moving Approaches
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_threshold = 0.5
    best_score = 0
    scores = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "accuracy":
            score = accuracy_score(y_true, y_pred)
        scores.append(score)

        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score, thresholds, scores


# Predict probabilities pada VALIDATION set
ngb_val_proba = ngb_model.predict_proba(X_val_scaled)[:, 1]
xgb_val_proba = xgb_pipeline.predict_proba(X_val)[:, 1]
rf_val_proba = rf_pipeline.predict_proba(X_val)[:, 1]

# Cari threshold optimal untuk setiap model
ngb_opt_thresh, ngb_opt_f1, ngb_thresholds, ngb_scores = find_optimal_threshold(y_val, ngb_val_proba)
xgb_opt_thresh, xgb_opt_f1, xgb_thresholds, xgb_scores = find_optimal_threshold(y_val, xgb_val_proba)
rf_opt_thresh, rf_opt_f1, rf_thresholds, rf_scores = find_optimal_threshold(y_val, rf_val_proba)

# Simpan threshold per model
optimal_thresholds = {
    "NGBoost": ngb_opt_thresh,
    "XGBoost": xgb_opt_thresh,
    "Random Forest": rf_opt_thresh
}

print("=" * 60)
print("THRESHOLD OPTIMIZATION RESULTS (Validation Set)")
print("=" * 60)
print(f"{'Model':<15} {'Optimal Threshold':<20} {'F1-Score (Val)':<15}")
print("-" * 50)
print(f"{'NGBoost':<15} {ngb_opt_thresh:<20.4f} {ngb_opt_f1:<15.4f}")
print(f"{'XGBoost':<15} {xgb_opt_thresh:<20.4f} {xgb_opt_f1:<15.4f}")
print(f"{'Random Forest':<15} {rf_opt_thresh:<20.4f} {rf_opt_f1:<15.4f}")

# Bandingkan dengan threshold default
print(f"\n--- Perbandingan dengan Threshold Default (0.5) ---")
ngb_f1_default = f1_score(y_val, (ngb_val_proba >= 0.5).astype(int))
xgb_f1_default = f1_score(y_val, (xgb_val_proba >= 0.5).astype(int))
rf_f1_default = f1_score(y_val, (rf_val_proba >= 0.5).astype(int))

print(f"{'Model':<15} {'F1(thr=0.5)':<15} {'F1(thr=opt)':<15} {'Improvement':<12}")
print("-" * 57)
print(f"{'NGBoost':<15} {ngb_f1_default:<15.4f} {ngb_opt_f1:<15.4f} {ngb_opt_f1-ngb_f1_default:<+12.4f}")
print(f"{'XGBoost':<15} {xgb_f1_default:<15.4f} {xgb_opt_f1:<15.4f} {xgb_opt_f1-xgb_f1_default:<+12.4f}")
print(f"{'Random Forest':<15} {rf_f1_default:<15.4f} {rf_opt_f1:<15.4f} {rf_opt_f1-rf_f1_default:<+12.4f}")

## Cell 8: Evaluasi pada Test Set (Dengan Threshold Optimal)

Evaluasi menggunakan threshold optimal dari Cell 7.
Perbandingan metrik antara threshold default (0.5) dan threshold optimal.

In [ ]:
# Fungsi untuk Expected Calibration Error (ECE)
def calculate_ece(y_true, y_prob, n_bins=10):
    """Calculate Expected Calibration Error with n bins."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (y_prob >= bin_boundaries[i]) & (y_prob < bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_acc = y_true[mask].mean()
            bin_conf = y_prob[mask].mean()
            ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(y_true)


# Predict probabilities pada TEST set
ngb_test_proba = ngb_model.predict_proba(X_test_scaled)[:, 1]
xgb_test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]
rf_test_proba = rf_pipeline.predict_proba(X_test)[:, 1]

# Evaluasi dengan threshold OPTIMAL
ngb_pred_opt = (ngb_test_proba >= optimal_thresholds["NGBoost"]).astype(int)
xgb_pred_opt = (xgb_test_proba >= optimal_thresholds["XGBoost"]).astype(int)
rf_pred_opt = (rf_test_proba >= optimal_thresholds["Random Forest"]).astype(int)

# Evaluasi dengan threshold DEFAULT (0.5)
ngb_pred_def = (ngb_test_proba >= 0.5).astype(int)
xgb_pred_def = (xgb_test_proba >= 0.5).astype(int)
rf_pred_def = (rf_test_proba >= 0.5).astype(int)

# Hitung metrik
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_true, y_prob),
        "NLL": log_loss(y_true, y_prob),
        "ECE": calculate_ece(y_true, y_prob)
    }

# Metrics dengan threshold optimal
metrics_ngb_opt = compute_metrics(y_test, ngb_pred_opt, ngb_test_proba)
metrics_xgb_opt = compute_metrics(y_test, xgb_pred_opt, xgb_test_proba)
metrics_rf_opt = compute_metrics(y_test, rf_pred_opt, rf_test_proba)

# Metrics dengan threshold default
metrics_ngb_def = compute_metrics(y_test, ngb_pred_def, ngb_test_proba)
metrics_xgb_def = compute_metrics(y_test, xgb_pred_def, xgb_test_proba)
metrics_rf_def = compute_metrics(y_test, rf_pred_def, rf_test_proba)

# Print tabel perbandingan
print("=" * 90)
print("EVALUASI TEST SET: Perbandingan Threshold Default vs Optimal")
print("=" * 90)
header = f"{'Metric':<12} {'NGBoost(0.5)':<14} {'NGBoost(opt)':<14} {'XGBoost(0.5)':<14} {'XGBoost(opt)':<14} {'RF(0.5)':<10} {'RF(opt)':<10}"
print(header)
print("-" * 90)

for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC", "NLL", "ECE"]:
    print(f"{metric:<12} "
          f"{metrics_ngb_def[metric]:<14.4f} {metrics_ngb_opt[metric]:<14.4f} "
          f"{metrics_xgb_def[metric]:<14.4f} {metrics_xgb_opt[metric]:<14.4f} "
          f"{metrics_rf_def[metric]:<14.4f} {metrics_rf_opt[metric]:<14.4f}")

print("\n" + "=" * 90)
print("EVALUASI TEST SET: Threshold Optimal")
print("=" * 90)
print(f"NGBoost threshold: {optimal_thresholds['NGBoost']:.4f}")
print(f"XGBoost threshold: {optimal_thresholds['XGBoost']:.4f}")
print(f"RF threshold: {optimal_thresholds['Random Forest']:.4f}")

# Confusion matrices
print("\n--- Confusion Matrices (Threshold Optimal) ---")
print("\nNGBoost:")
print(confusion_matrix(y_test, ngb_pred_opt))
print("\nXGBoost:")
print(confusion_matrix(y_test, xgb_pred_opt))
print("\nRandom Forest:")
print(confusion_matrix(y_test, rf_pred_opt))

## Cell 9: McNemar's Test

Uji statistik McNemar untuk membandingkan performa antar model.
Menggunakan prediksi dengan threshold OPTIMAL.

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    """Perform McNemar's test between two classifiers."""
    # Contingency table
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    b = ((correct_a) & (~correct_b)).sum()  # A right, B wrong
    c = ((~correct_a) & (correct_b)).sum()  # A wrong, B right

    # McNemar's test with continuity correction
    if (b + c) == 0:
        statistic = 0
        p_value = 1.0
    else:
        statistic = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1 - chi2.cdf(statistic, df=1)

    return statistic, p_value, b, c


print("=" * 60)
print("McNEMAR'S TEST (Threshold Optimal)")
print("=" * 60)
print()

# NGBoost vs XGBoost
stat, pval, b, c = mcnemar_test(y_test, ngb_pred_opt, xgb_pred_opt)
print(f"NGBoost vs XGBoost:")
print(f"  Statistic = {stat:.4f}, p-value = {pval:.4f}")
print(f"  b (NGBoost correct, XGBoost wrong) = {b}")
print(f"  c (NGBoost wrong, XGBoost correct) = {c}")
print(f"  Significant (p<0.05): {'Yes' if pval < 0.05 else 'No'}")
print()

# NGBoost vs RF
stat, pval, b, c = mcnemar_test(y_test, ngb_pred_opt, rf_pred_opt)
print(f"NGBoost vs Random Forest:")
print(f"  Statistic = {stat:.4f}, p-value = {pval:.4f}")
print(f"  b (NGBoost correct, RF wrong) = {b}")
print(f"  c (NGBoost wrong, RF correct) = {c}")
print(f"  Significant (p<0.05): {'Yes' if pval < 0.05 else 'No'}")
print()

# XGBoost vs RF
stat, pval, b, c = mcnemar_test(y_test, xgb_pred_opt, rf_pred_opt)
print(f"XGBoost vs Random Forest:")
print(f"  Statistic = {stat:.4f}, p-value = {pval:.4f}")
print(f"  b (XGBoost correct, RF wrong) = {b}")
print(f"  c (XGBoost wrong, RF correct) = {c}")
print(f"  Significant (p<0.05): {'Yes' if pval < 0.05 else 'No'}")

## Cell 10: Uncertainty Zone Analysis

Membagi prediksi probabilitas ke 5 zona untuk menganalisis kepercayaan model.
Evaluasi menggunakan threshold optimal.

In [ ]:
def uncertainty_zone_analysis(y_true, y_prob, model_name, threshold):
    """Analyze predictions by probability zones."""
    zones = [
        ("Very Low (0.0-0.2)", 0.0, 0.2),
        ("Low (0.2-0.4)", 0.2, 0.4),
        ("Medium (0.4-0.6)", 0.4, 0.6),
        ("High (0.6-0.8)", 0.6, 0.8),
        ("Very High (0.8-1.0)", 0.8, 1.0)
    ]

    print(f"\n{'Zone':<22} {'N Samples':<12} {'Accuracy':<12} {'Avg Prob':<12}")
    print("-" * 58)

    for zone_name, low, high in zones:
        if high == 1.0:
            mask = (y_prob >= low) & (y_prob <= high)
        else:
            mask = (y_prob >= low) & (y_prob < high)

        n_samples = mask.sum()
        if n_samples > 0:
            y_pred_zone = (y_prob[mask] >= threshold).astype(int)
            acc = accuracy_score(y_true[mask], y_pred_zone)
            avg_prob = y_prob[mask].mean()
            print(f"{zone_name:<22} {n_samples:<12} {acc:<12.4f} {avg_prob:<12.4f}")
        else:
            print(f"{zone_name:<22} {0:<12} {'N/A':<12} {'N/A':<12}")


print("=" * 60)
print("UNCERTAINTY ZONE ANALYSIS (Test Set, Threshold Optimal)")
print("=" * 60)

print(f"\n--- NGBoost (threshold={optimal_thresholds['NGBoost']:.4f}) ---")
uncertainty_zone_analysis(y_test.values, ngb_test_proba, "NGBoost", optimal_thresholds["NGBoost"])

print(f"\n--- XGBoost (threshold={optimal_thresholds['XGBoost']:.4f}) ---")
uncertainty_zone_analysis(y_test.values, xgb_test_proba, "XGBoost", optimal_thresholds["XGBoost"])

print(f"\n--- Random Forest (threshold={optimal_thresholds['Random Forest']:.4f}) ---")
uncertainty_zone_analysis(y_test.values, rf_test_proba, "Random Forest", optimal_thresholds["Random Forest"])

## Cell 11: Visualisasi - Threshold Optimization Plot

Plot F1-Score vs Threshold untuk ketiga model dengan garis vertikal di threshold optimal.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(ngb_thresholds, ngb_scores, label=f"NGBoost (opt={ngb_opt_thresh:.2f})", color="blue", linewidth=2)
ax.plot(xgb_thresholds, xgb_scores, label=f"XGBoost (opt={xgb_opt_thresh:.2f})", color="red", linewidth=2)
ax.plot(rf_thresholds, rf_scores, label=f"Random Forest (opt={rf_opt_thresh:.2f})", color="green", linewidth=2)

# Vertical dashed lines at optimal thresholds
ax.axvline(x=ngb_opt_thresh, color="blue", linestyle="--", alpha=0.7)
ax.axvline(x=xgb_opt_thresh, color="red", linestyle="--", alpha=0.7)
ax.axvline(x=rf_opt_thresh, color="green", linestyle="--", alpha=0.7)

# Default threshold reference
ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.5, label="Default (0.5)")

ax.set_xlabel("Threshold", fontsize=12)
ax.set_ylabel("F1-Score", fontsize=12)
ax.set_title("Threshold Optimization: F1-Score vs Decision Threshold", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 0.9)

plt.tight_layout()
plt.savefig("figures/threshold_optimization_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/threshold_optimization_v3.png")

## Cell 12: Visualisasi - Calibration Curves

Plot calibration curves untuk mengevaluasi kalibrasi probabilitas model.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Calibration curves
for name, y_prob, color in [
    ("NGBoost", ngb_test_proba, "blue"),
    ("XGBoost", xgb_test_proba, "red"),
    ("Random Forest", rf_test_proba, "green")
]:
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, y_prob, n_bins=10
    )
    ax.plot(mean_predicted_value, fraction_of_positives,
            marker="o", label=name, color=color, linewidth=2)

# Perfect calibration line
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")

ax.set_xlabel("Mean Predicted Probability", fontsize=12)
ax.set_ylabel("Fraction of Positives", fontsize=12)
ax.set_title("Calibration Curves", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/calibration_curves_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/calibration_curves_v3.png")

## Cell 13: Visualisasi - ROC Curves

Plot ROC curves untuk ketiga model.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

for name, y_prob, color in [
    ("NGBoost", ngb_test_proba, "blue"),
    ("XGBoost", xgb_test_proba, "red"),
    ("Random Forest", rf_test_proba, "green")
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", color=color, linewidth=2)

# Random classifier line
ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/roc_curves_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/roc_curves_v3.png")

## Cell 14: Visualisasi - KDE Probability Distributions

Distribusi probabilitas prediksi per kelas dengan garis vertikal di threshold optimal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_info = [
    ("NGBoost", ngb_test_proba, optimal_thresholds["NGBoost"], "blue"),
    ("XGBoost", xgb_test_proba, optimal_thresholds["XGBoost"], "red"),
    ("Random Forest", rf_test_proba, optimal_thresholds["Random Forest"], "green")
]

for ax, (name, proba, thresh, color) in zip(axes, models_info):
    # KDE for class 0
    sns.kdeplot(proba[y_test == 0], ax=ax, label="Class 0 (Not Potable)",
               color="orange", fill=True, alpha=0.3)
    # KDE for class 1
    sns.kdeplot(proba[y_test == 1], ax=ax, label="Class 1 (Potable)",
               color="blue", fill=True, alpha=0.3)

    # Optimal threshold line
    ax.axvline(x=thresh, color="red", linestyle="--", linewidth=2,
              label=f"Optimal Threshold ({thresh:.2f})")

    ax.set_xlabel("Predicted Probability", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"{name}", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("KDE Probability Distributions (with Optimal Threshold)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/kde_distributions_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/kde_distributions_v3.png")

## Cell 15: Visualisasi - Confusion Matrices

Heatmap confusion matrix untuk ketiga model (menggunakan threshold optimal).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

predictions = [
    ("NGBoost", ngb_pred_opt),
    ("XGBoost", xgb_pred_opt),
    ("Random Forest", rf_pred_opt)
]

for ax, (name, y_pred) in zip(axes, predictions):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
               xticklabels=["Not Potable", "Potable"],
               yticklabels=["Not Potable", "Potable"])
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)
    thresh_val = optimal_thresholds[name]
    ax.set_title(f"{name} (thr={thresh_val:.2f})", fontsize=13)

plt.suptitle("Confusion Matrices (Threshold Optimal)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/confusion_matrices_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/confusion_matrices_v3.png")

## Cell 16: Visualisasi - Feature Importance

Feature importance untuk ketiga model:
- NGBoost: Permutation Importance
- XGBoost: Built-in feature_importances_
- Random Forest: Built-in feature_importances_

In [ ]:
# NGBoost: Permutation Importance
print("Calculating NGBoost permutation importance...")
perm_imp = permutation_importance(
    ngb_model, X_test_scaled, y_test,
    n_repeats=10, random_state=42, scoring="f1"
)
ngb_importances = perm_imp.importances_mean

# XGBoost: Built-in feature importances
xgb_importances = xgb_pipeline.named_steps["classifier"].feature_importances_

# RF: Built-in feature importances
rf_importances = rf_pipeline.named_steps["classifier"].feature_importances_

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

importances_data = [
    ("NGBoost (Permutation)", ngb_importances, "blue"),
    ("XGBoost (Built-in)", xgb_importances, "red"),
    ("Random Forest (Built-in)", rf_importances, "green")
]

for ax, (name, importances, color) in zip(axes, importances_data):
    sorted_idx = np.argsort(importances)
    ax.barh(range(len(feature_names)), importances[sorted_idx], color=color, alpha=0.7)
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx])
    ax.set_xlabel("Importance", fontsize=11)
    ax.set_title(name, fontsize=12)
    ax.grid(True, alpha=0.3, axis="x")

plt.suptitle("Feature Importance Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/feature_importance_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/feature_importance_v3.png")

## Cell 17: Stratified 5-Fold Cross-Validation (Robustness)

Validasi silang pada seluruh dataset untuk menguji robustness model.
Dalam setiap fold: apply pipeline (impute + scale + SMOTE) pada training fold.
Threshold dioptimasi menggunakan inner validation split dari training fold.

In [ ]:
print("=" * 60)
print("STRATIFIED 5-FOLD CROSS-VALIDATION")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage for results
cv_results = {
    "NGBoost": {"accuracy": [], "f1": []},
    "XGBoost": {"accuracy": [], "f1": []},
    "Random Forest": {"accuracy": [], "f1": []}
}

X_full = df.drop("Potability", axis=1).values
y_full = df["Potability"].values

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y_full)):
    print(f"\nFold {fold + 1}/5...")

    X_cv_train, X_cv_test = X_full[train_idx], X_full[test_idx]
    y_cv_train, y_cv_test = y_full[train_idx], y_full[test_idx]

    # Inner split for threshold optimization (80/20 from training fold)
    X_cv_tr, X_cv_val, y_cv_tr, y_cv_val = train_test_split(
        X_cv_train, y_cv_train, test_size=0.2, random_state=42, stratify=y_cv_train
    )

    # Preprocessing for NGBoost
    imp_cv = SimpleImputer(strategy="median")
    X_cv_tr_imp = imp_cv.fit_transform(X_cv_tr)
    X_cv_val_imp = imp_cv.transform(X_cv_val)
    X_cv_test_imp = imp_cv.transform(X_cv_test)

    sc_cv = MinMaxScaler()
    X_cv_tr_sc = sc_cv.fit_transform(X_cv_tr_imp)
    X_cv_val_sc = sc_cv.transform(X_cv_val_imp)
    X_cv_test_sc = sc_cv.transform(X_cv_test_imp)

    sm_cv = SMOTE(random_state=42)
    X_cv_tr_res, y_cv_tr_res = sm_cv.fit_resample(X_cv_tr_sc, y_cv_tr)

    # --- NGBoost ---
    ngb_cv = NGBClassifier(
        Dist=Bernoulli, n_estimators=300, learning_rate=0.05,
        minibatch_frac=0.8, col_sample=0.8, random_state=42,
        Base=DecisionTreeRegressor(max_depth=4), verbose=False
    )
    ngb_cv.fit(X_cv_tr_res, y_cv_tr_res, X_val=X_cv_val_sc, Y_val=y_cv_val)

    # Find optimal threshold on inner val
    ngb_cv_val_prob = ngb_cv.predict_proba(X_cv_val_sc)[:, 1]
    ngb_cv_thresh, _, _, _ = find_optimal_threshold(y_cv_val, ngb_cv_val_prob)

    ngb_cv_test_prob = ngb_cv.predict_proba(X_cv_test_sc)[:, 1]
    ngb_cv_pred = (ngb_cv_test_prob >= ngb_cv_thresh).astype(int)
    cv_results["NGBoost"]["accuracy"].append(accuracy_score(y_cv_test, ngb_cv_pred))
    cv_results["NGBoost"]["f1"].append(f1_score(y_cv_test, ngb_cv_pred))

    # --- XGBoost (via ImbPipeline) ---
    xgb_cv_pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("smote", SMOTE(random_state=42)),
        ("classifier", XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric="logloss", use_label_encoder=False, verbosity=0
        ))
    ])
    xgb_cv_pipe.fit(X_cv_tr, y_cv_tr)

    # Threshold on inner val
    xgb_cv_val_prob = xgb_cv_pipe.predict_proba(X_cv_val)[:, 1]
    xgb_cv_thresh, _, _, _ = find_optimal_threshold(y_cv_val, xgb_cv_val_prob)

    xgb_cv_test_prob = xgb_cv_pipe.predict_proba(X_cv_test)[:, 1]
    xgb_cv_pred = (xgb_cv_test_prob >= xgb_cv_thresh).astype(int)
    cv_results["XGBoost"]["accuracy"].append(accuracy_score(y_cv_test, xgb_cv_pred))
    cv_results["XGBoost"]["f1"].append(f1_score(y_cv_test, xgb_cv_pred))

    # --- Random Forest (via ImbPipeline) ---
    rf_cv_pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("smote", SMOTE(random_state=42)),
        ("classifier", RandomForestClassifier(n_estimators=300, random_state=42))
    ])
    rf_cv_pipe.fit(X_cv_tr, y_cv_tr)

    # Threshold on inner val
    rf_cv_val_prob = rf_cv_pipe.predict_proba(X_cv_val)[:, 1]
    rf_cv_thresh, _, _, _ = find_optimal_threshold(y_cv_val, rf_cv_val_prob)

    rf_cv_test_prob = rf_cv_pipe.predict_proba(X_cv_test)[:, 1]
    rf_cv_pred = (rf_cv_test_prob >= rf_cv_thresh).astype(int)
    cv_results["Random Forest"]["accuracy"].append(accuracy_score(y_cv_test, rf_cv_pred))
    cv_results["Random Forest"]["f1"].append(f1_score(y_cv_test, rf_cv_pred))

# Print results
print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS (5-Fold, with Threshold Optimization)")
print("=" * 60)
print(f"{'Model':<15} {'Accuracy (mean +/- std)':<25} {'F1-Score (mean +/- std)':<25}")
print("-" * 65)
for model_name in ["NGBoost", "XGBoost", "Random Forest"]:
    acc_mean = np.mean(cv_results[model_name]["accuracy"])
    acc_std = np.std(cv_results[model_name]["accuracy"])
    f1_mean = np.mean(cv_results[model_name]["f1"])
    f1_std = np.std(cv_results[model_name]["f1"])
    print(f"  {model_name:<15} {acc_mean:.4f} +/- {acc_std:.4f}        {f1_mean:.4f} +/- {f1_std:.4f}")

## Cell 18: Perbandingan V2 vs V3

Perbandingan performa antara:
- V2: Threshold default (0.5)
- V3: Threshold optimal (dari validation set)

In [ ]:
print("=" * 80)
print("PERBANDINGAN V2 (thr=0.5) vs V3 (thr=optimal)")
print("=" * 80)

# V2 results (threshold = 0.5)
v2_ngb_acc = metrics_ngb_def["Accuracy"]
v2_xgb_acc = metrics_xgb_def["Accuracy"]
v2_rf_acc = metrics_rf_def["Accuracy"]
v2_ngb_f1 = metrics_ngb_def["F1-Score"]
v2_xgb_f1 = metrics_xgb_def["F1-Score"]
v2_rf_f1 = metrics_rf_def["F1-Score"]

# V3 results (threshold = optimal)
v3_ngb_acc = metrics_ngb_opt["Accuracy"]
v3_xgb_acc = metrics_xgb_opt["Accuracy"]
v3_rf_acc = metrics_rf_opt["Accuracy"]
v3_ngb_f1 = metrics_ngb_opt["F1-Score"]
v3_xgb_f1 = metrics_xgb_opt["F1-Score"]
v3_rf_f1 = metrics_rf_opt["F1-Score"]

header = "Config".ljust(16) + "NGBoost_Acc".ljust(12) + "XGBoost_Acc".ljust(12) + "RF_Acc".ljust(10)
header += "NGBoost_F1".ljust(12) + "XGBoost_F1".ljust(12) + "RF_F1".ljust(10)
print(header)
print("-" * 84)

row_v2 = "V2 (thr=0.5)".ljust(16)
row_v2 += f"{v2_ngb_acc:<12.4f}{v2_xgb_acc:<12.4f}{v2_rf_acc:<10.4f}"
row_v2 += f"{v2_ngb_f1:<12.4f}{v2_xgb_f1:<12.4f}{v2_rf_f1:<10.4f}"
print(row_v2)

row_v3 = "V3 (thr=opt)".ljust(16)
row_v3 += f"{v3_ngb_acc:<12.4f}{v3_xgb_acc:<12.4f}{v3_rf_acc:<10.4f}"
row_v3 += f"{v3_ngb_f1:<12.4f}{v3_xgb_f1:<12.4f}{v3_rf_f1:<10.4f}"
print(row_v3)

row_d = "Delta".ljust(16)
row_d += f"{v3_ngb_acc-v2_ngb_acc:<+12.4f}{v3_xgb_acc-v2_xgb_acc:<+12.4f}{v3_rf_acc-v2_rf_acc:<+10.4f}"
row_d += f"{v3_ngb_f1-v2_ngb_f1:<+12.4f}{v3_xgb_f1-v2_xgb_f1:<+12.4f}{v3_rf_f1-v2_rf_f1:<+10.4f}"
print(row_d)

print(f"\nOptimal Thresholds:")
print(f"  NGBoost: {optimal_thresholds['NGBoost']:.4f}")
print(f"  XGBoost: {optimal_thresholds['XGBoost']:.4f}")
print(f"  Random Forest: {optimal_thresholds['Random Forest']:.4f}")

## Cell 19: Ringkasan Hasil

Ringkasan lengkap semua hasil Pipeline V3.

In [ ]:
print("=" * 70)
print("RINGKASAN HASIL PIPELINE V3")
print("NGBoost + DiCE: Water Quality Classification")
print("Median Imputation + MinMax + SMOTE + Threshold Optimization")
print("=" * 70)

print("\n1. DATASET:")
print(f"   - Total samples: {len(df)}")
feat_str = ", ".join(feature_names)
print(f"   - Features: {len(feature_names)} ({feat_str})")
print(f"   - Train/Val/Test split: {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}")

print("\n2. PREPROCESSING:")
print("   - Missing value imputation: Median (fit on train only)")
print("   - Feature scaling: MinMaxScaler (fit on train only)")
print(f"   - SMOTE: {X_train_scaled.shape[0]} -> {X_train_resampled.shape[0]} (train only)")

print("\n3. THRESHOLD OPTIMIZATION (Validation Set):")
for model_name, thresh in optimal_thresholds.items():
    print(f"   - {model_name}: {thresh:.4f}")

print("\n4. TEST SET RESULTS (Threshold Optimal):")
header_r = "   " + "Model".ljust(15) + "Accuracy".ljust(10) + "F1-Score".ljust(10)
header_r += "AUC-ROC".ljust(10) + "NLL".ljust(10) + "ECE".ljust(10)
print(header_r)
print("   " + "-"*65)
for mname, mdict in [("NGBoost", metrics_ngb_opt), ("XGBoost", metrics_xgb_opt), ("Random Forest", metrics_rf_opt)]:
    row = f"   {mname:<15}{mdict['Accuracy']:<10.4f}{mdict['F1-Score']:<10.4f}"
    row += f"{mdict['AUC-ROC']:<10.4f}{mdict['NLL']:<10.4f}{mdict['ECE']:<10.4f}"
    print(row)

print("\n5. IMPROVEMENT OVER V2 (Delta F1-Score):")
print(f"   - NGBoost: {v3_ngb_f1 - v2_ngb_f1:+.4f}")
print(f"   - XGBoost: {v3_xgb_f1 - v2_xgb_f1:+.4f}")
print(f"   - Random Forest: {v3_rf_f1 - v2_rf_f1:+.4f}")

print("\n6. CROSS-VALIDATION (5-Fold with Threshold Optimization):")
for model_name in ["NGBoost", "XGBoost", "Random Forest"]:
    acc_mean = np.mean(cv_results[model_name]["accuracy"])
    acc_std = np.std(cv_results[model_name]["accuracy"])
    f1_mean = np.mean(cv_results[model_name]["f1"])
    f1_std = np.std(cv_results[model_name]["f1"])
    print(f"   - {model_name}: Acc={acc_mean:.4f}+/-{acc_std:.4f}, F1={f1_mean:.4f}+/-{f1_std:.4f}")

print("\n" + "=" * 70)
print("Pipeline V3 Complete!")
print("=" * 70)